# Long method Smell Detection
## Consolidated Notebook: Rules → Traditional ML → TabPFN → Ensemble
### GroupKFold cross-validation | All methods | Deployment-ready

| Input | Path |
|---|---|
| Industrial | `drive/MyDrive/codesmell/data/smellycodefull.csv` |
| Student    | `drive/MyDrive/codesmell/data/student_features_with_labels.csv` |
| Output     | `drive/MyDrive/codesmellimprovements/god_class/` |

label_data_class ,label_long_method ,label_feature_envy

In [ ]:
# ── Cell 1: Install ──────────────────────────────────────────────────────────
!pip install xgboost imbalanced-learn scikit-learn joblib pandas numpy -q
!pip install tabpfn -q
!pip install lightgbm -q
print("All packages installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.8/229.8 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
All packages installed.


In [ ]:
# ── Cell 2: Imports + Drive mount + Global storage ───────────────────────────
import os, json, warnings, joblib, pickle
import numpy  as np
import pandas as pd
import xgboost as xgb
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Set TabPFN token before any tabpfn import
os.environ["TABPFN_TOKEN"] = ""
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from sklearn.ensemble          import RandomForestClassifier
from sklearn.tree              import DecisionTreeClassifier
from sklearn.linear_model      import LogisticRegression
from sklearn.preprocessing     import StandardScaler
from sklearn.model_selection   import GroupKFold, StratifiedKFold
from sklearn.metrics           import (
    f1_score, precision_score, recall_score,
    average_precision_score, confusion_matrix,
    precision_recall_curve
)
from imblearn.combine          import SMOTETomek, SMOTEENN
from imblearn.over_sampling    import SMOTE
from imblearn.pipeline         import Pipeline as ImbPipeline

# ── Global OOF accumulator ────────────────────────────────────────────────────
# Each method cell populates:
#   ALL_RESULTS[model_name] = {'prob': np.array, 'pred_05': np.array, 'category': str}
ALL_RESULTS = {}

print(f"XGBoost {xgb.__version__}  ready")
print("ALL_RESULTS dict initialised — ready to accumulate OOF predictions.")

Mounted at /content/drive
XGBoost 3.2.0  ready
ALL_RESULTS dict initialised — ready to accumulate OOF predictions.


In [ ]:
# ── Cell 3: Paths, constants, feature columns ────────────────────────────────
BASE      = '/content/drive/MyDrive/codesmell'
IND_DATA  = f'{BASE}/data/smellycodefull.csv'
STUD_DATA = f'{BASE}/data/student_features_with_labels.csv'

OUT_BASE   = '/content/drive/MyDrive/CodeSmell_Stage3/long_method'
P1_MODELS  = f'{OUT_BASE}/phase1/models'
P1_RESULTS = f'{OUT_BASE}/phase1/results'
PA_MODELS  = f'{OUT_BASE}/modelA/models'
PA_RESULTS = f'{OUT_BASE}/modelA/results'
PT_MODELS  = f'{OUT_BASE}/trAda/models'
PT_RESULTS = f'{OUT_BASE}/trAda/results'
DEPLOY_DIR = f'{OUT_BASE}/deployment'

for d in [P1_MODELS, P1_RESULTS, PA_MODELS, PA_RESULTS,
          PT_MODELS, PT_RESULTS, DEPLOY_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

FEATURE_COLS = [
    'Logical Lines', 'Distinct Operators', 'Distinct Operands',
    'Total Operators', 'Total Operands', 'Cyclomatic Complexity',
]
TARGET      = 'label_long_method'
IND_LABEL   = 'Long method'
# 'Long method', 'God class', 'Feature envy', 'Data class'
IND_PROJECT = 'Project'

print("Paths configured.")
print(f"  Industrial data : {IND_DATA}")
print(f"  Student data    : {STUD_DATA}")
print(f"  Output base     : {OUT_BASE}")

Paths configured.
  Industrial data : /content/drive/MyDrive/codesmell/data/smellycodefull.csv
  Student data    : /content/drive/MyDrive/codesmell/data/student_features_with_labels.csv
  Output base     : /content/drive/MyDrive/CodeSmell_Stage3/long_method


In [ ]:
# ── Cell 4: Load industrial + student data ───────────────────────────────────
# ── Industrial ────────────────────────────────────────────────────────────────
df_ind = pd.read_csv(IND_DATA)
missing_ind = [c for c in FEATURE_COLS + [IND_LABEL, IND_PROJECT] if c not in df_ind.columns]
if missing_ind:
    raise ValueError(f"Industrial data missing columns: {missing_ind}")

df_ind[FEATURE_COLS] = df_ind[FEATURE_COLS].fillna(0)
X_ind = df_ind[FEATURE_COLS].values
y_ind = df_ind[IND_LABEL].values.astype(int)
g_ind = df_ind[IND_PROJECT].values
pos_i = int(y_ind.sum())
print(f"Industrial : {df_ind.shape[0]:,} samples | {pos_i} Long method ({pos_i/len(y_ind)*100:.1f}%)")
print(f"  Projects : {df_ind[IND_PROJECT].nunique()}")

# ── Student ────────────────────────────────────────────────────────────────────
df_stu = pd.read_csv(STUD_DATA)
missing_stu = [c for c in FEATURE_COLS + [TARGET, 'project'] if c not in df_stu.columns]
if missing_stu:
    raise ValueError(f"Student data missing columns: {missing_stu}")

df_stu = df_stu.dropna(subset=FEATURE_COLS + [TARGET]).reset_index(drop=True)
df_stu[TARGET] = df_stu[TARGET].astype(int)

X_all      = df_stu[FEATURE_COLS].values.astype(float)
y_all      = df_stu[TARGET].values.astype(int)
groups_all = df_stu['project'].values
n_all      = len(df_stu)
n_pos_all  = int(y_all.sum())
n_repos    = df_stu['project'].nunique()

print(f"\nStudent    : {n_all:,} samples | {n_pos_all} Long method ({n_pos_all/n_all*100:.1f}%)")
print(f"  Projects : {n_repos}")

# ── Per-repo summary ──────────────────────────────────────────────────────────
print("\nPer-repo positives:")
for repo, grp in df_stu.groupby('project'):
    p = int(grp[TARGET].sum())
    print(f"  {repo:45s}  {p:2d}  {'█'*p}")

# ── GroupKFold setup (shared by all subsequent cells) ─────────────────────────
N_SPLITS = min(5, n_repos)
while N_SPLITS > 2 and n_pos_all / N_SPLITS < 1.5:
    N_SPLITS -= 1
gkf = GroupKFold(n_splits=N_SPLITS)
print(f"\nGroupKFold: n_splits={N_SPLITS}  total_pos={n_pos_all}  ~pos/fold={n_pos_all/N_SPLITS:.1f}")

Industrial : 107,554 samples | 1566 Long method (1.5%)
  Projects : 89

Student    : 3,266 samples | 341 Long method (10.4%)
  Projects : 821

Per-repo positives:
  20+Java-Question-lab-work                       0  
  4sum.java                                       0  
  A.java                                          0  
  AES.java                                        0  
  AI Riddle Game                                  0  
  API-Lesson1                                     0  
  ATM                                             0  
  ATM System                                      0  
  ATM interface                                   0  
  ATM-Bankomat                                    0  
  ATM2-Bankomat                                   0  
  AbstractClassesAndMethods                       0  
  Abstraction & Interfaces.java                   0  
  AccessModifiers                                 0  
  Addhar entry                                    0  
  Address Book             

In [ ]:
# ── Cell 5: Model_I — Train on industrial data, save .joblib, extract booster ─
# Used as: (a) zero-shot transfer baseline on student data
#          (b) starting checkpoint for TrAdaBoost (Cell 8)
print("="*72)
print("MODEL_I: Industrial data  |  GroupKFold/StratifiedKFold + SMOTETomek")
print("="*72)

n_proj = df_ind[IND_PROJECT].nunique()
if n_proj >= 5:
    cv_ind     = GroupKFold(n_splits=min(5, n_proj))
    split_args = (X_ind, y_ind, g_ind)
    cv_desc    = f"GroupKFold(n={min(5,n_proj)})"
else:
    cv_ind     = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    split_args = (X_ind, y_ind)
    cv_desc    = "StratifiedKFold(n=5)"

def make_ind_pipe(clf):
    return ImbPipeline([
        ('scaler',   StandardScaler()),
        ('resample', SMOTETomek(random_state=RANDOM_STATE)),
        ('clf',      clf),
    ])

pipelines_I = {
    'LogisticRegression': make_ind_pipe(
        LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)),
    'DecisionTree':       make_ind_pipe(
        DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=RANDOM_STATE)),
    'RandomForest':       make_ind_pipe(
        RandomForestClassifier(n_estimators=100, class_weight='balanced',
                               random_state=RANDOM_STATE, n_jobs=-1)),
    'XGBoost':            make_ind_pipe(
        xgb.XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                          eval_metric='logloss', random_state=RANDOM_STATE,
                          n_jobs=-1, verbosity=0)),
}

print(f"CV: {cv_desc}")
print(f"\n{'Model':22s}  {'F1':>6}  {'Prec':>6}  {'Rec':>6}  {'PR-AUC':>7}")
print("-"*56)

fitted_I = {}
for name, pipe in pipelines_I.items():
    yp  = np.zeros(len(y_ind), dtype=int)
    prb = np.zeros(len(y_ind))
    for tr, va in cv_ind.split(*split_args):
        Xtr, Xva, ytr = X_ind[tr], X_ind[va], y_ind[tr]
        if ytr.sum() < 2:
            continue
        pipe.fit(Xtr, ytr)
        yp[va]  = pipe.predict(Xva)
        prb[va] = pipe.predict_proba(Xva)[:, 1]
    pipe.fit(X_ind, y_ind)
    fitted_I[name] = pipe
    f1   = f1_score(y_ind, yp, zero_division=0)
    prec = precision_score(y_ind, yp, zero_division=0)
    rec  = recall_score(y_ind, yp, zero_division=0)
    pr   = average_precision_score(y_ind, prb)
    print(f"  {name:20s}  {f1:6.3f}  {prec:6.3f}  {rec:6.3f}  {pr:7.3f}")
    joblib.dump(pipe, f"{P1_MODELS}/model_I_long_class_{name}.joblib")

pd.DataFrame([{
    'model': n, 'f1': round(f1_score(y_ind,np.zeros(len(y_ind),dtype=int)),3)
} for n in fitted_I]).to_csv(f"{P1_RESULTS}/phase1_results.csv", index=False)

# ── Extract XGBoost booster + industrial scaler (needed for TrAda in Cell 8) ─
model_I_xgb = fitted_I['XGBoost']
booster_I   = model_I_xgb.named_steps['clf'].get_booster()
ind_scaler  = model_I_xgb.named_steps['scaler']
joblib.dump(ind_scaler, f"{P1_MODELS}/ind_scaler.joblib")

print(f"\nModel_I saved → {P1_MODELS}")
print("booster_I + ind_scaler ready (used by TrAdaBoost in Cell 8)")

MODEL_I: Industrial data  |  GroupKFold/StratifiedKFold + SMOTETomek
CV: GroupKFold(n=5)

Model                       F1    Prec     Rec   PR-AUC
--------------------------------------------------------
  LogisticRegression     0.055   0.029   0.612    0.031
  DecisionTree           0.116   0.062   0.798    0.082
  RandomForest           0.236   0.179   0.349    0.125
  XGBoost                0.183   0.105   0.711    0.148

Model_I saved → /content/drive/MyDrive/CodeSmell_Stage3/long_method/phase1/models
booster_I + ind_scaler ready (used by TrAdaBoost in Cell 8)


In [ ]:
# ── Cell 6: Rule-based baselines — GroupKFold OOF on student data ────────────
# Rules derived from feature importance analysis (Logical Lines dominates).
# Rule1  — Single:  Logical Lines > P75
# Rule2  — AND:     LL > P75 AND Total Operands > P75 AND Distinct Operands > P75
# Rule3  — OR:      any feature > P90
# Rule4  — Scoring: LL>P75 (+3), TotalOp>P75 (+2), DistOp>P75 (+1),
#                   TotalOpR>P75 (+1), CC>P75 (+1) → predict if score >= 3
print("="*72)
print("RULE-BASED BASELINES")
print("="*72)

FEAT_IDX = {f: i for i, f in enumerate(FEATURE_COLS)}
ll_idx   = FEAT_IDX['Logical Lines']
do_idx   = FEAT_IDX['Distinct Operands']
toi_idx  = FEAT_IDX['Total Operators']
top_idx  = FEAT_IDX['Total Operands']
cc_idx   = FEAT_IDX['Cyclomatic Complexity']

rule_preds = {
    'Rule1_Single':  np.zeros(n_all, dtype=int),
    'Rule2_AND':     np.zeros(n_all, dtype=int),
    'Rule3_OR':      np.zeros(n_all, dtype=int),
    'Rule4_Score':   np.zeros(n_all, dtype=int),
}

for _, (tr_idx, va_idx) in enumerate(gkf.split(X_all, y_all, groups_all)):
    X_tr, X_va = X_all[tr_idx], X_all[va_idx]
    p75 = np.percentile(X_tr, 75, axis=0)
    p90 = np.percentile(X_tr, 90, axis=0)

    rule_preds['Rule1_Single'][va_idx] = (
        X_va[:, ll_idx] > p75[ll_idx]).astype(int)

    rule_preds['Rule2_AND'][va_idx] = (
        (X_va[:, ll_idx]  > p75[ll_idx]) &
        (X_va[:, top_idx] > p75[top_idx]) &
        (X_va[:, do_idx]  > p75[do_idx])
    ).astype(int)

    rule_preds['Rule3_OR'][va_idx] = np.any(
        X_va > p90[np.newaxis, :], axis=1).astype(int)

    score = (
        (X_va[:, ll_idx]  > p75[ll_idx]).astype(int)  * 3 +
        (X_va[:, top_idx] > p75[top_idx]).astype(int) * 2 +
        (X_va[:, do_idx]  > p75[do_idx]).astype(int)  * 1 +
        (X_va[:, toi_idx] > p75[toi_idx]).astype(int) * 1 +
        (X_va[:, cc_idx]  > p75[cc_idx]).astype(int)  * 1
    )
    rule_preds['Rule4_Score'][va_idx] = (score >= 3).astype(int)

print(f"\n{'Rule':20s}  {'F1':>6}  {'Prec':>6}  {'Rec':>6}  {'PR-AUC':>7}  {'TP':>3}  {'FP':>3}")
print("-"*62)

for rname, rpred in rule_preds.items():
    f1   = f1_score(y_all, rpred, zero_division=0)
    prec = precision_score(y_all, rpred, zero_division=0)
    rec  = recall_score(y_all, rpred, zero_division=0)
    pr   = average_precision_score(y_all, rpred.astype(float))
    cm   = confusion_matrix(y_all, rpred)
    tp, fp = cm[1,1], cm[0,1]
    print(f"  {rname:18s}  {f1:6.3f}  {prec:6.3f}  {rec:6.3f}  {pr:7.3f}  {tp:3d}  {fp:3d}")
    # Rules have no continuous probability; use binary pred as proxy
    ALL_RESULTS[rname] = {
        'prob':     rpred.astype(float),
        'pred_05':  rpred,
        'category': 'Rule'
    }

print(f"\nRule baselines → ALL_RESULTS ({len(ALL_RESULTS)} entries)")

RULE-BASED BASELINES

Rule                      F1    Prec     Rec   PR-AUC   TP   FP
--------------------------------------------------------------
  Rule1_Single         0.418   0.297   0.707    0.240  241  571
  Rule2_AND            0.390   0.315   0.510    0.212  174  378
  Rule3_OR             0.354   0.283   0.475    0.189  162  411
  Rule4_Score          0.395   0.270   0.733    0.226  250  676

Rule baselines → ALL_RESULTS (4 entries)


In [ ]:
# ── Cell 7: Traditional ML — Model_A (student) + Model_I zero-shot ────────────
# Model_I  → pre-trained on industrial, applied as-is to student (zero-shot)
# Model_A  → trained from scratch on student train folds (LR / DT / RF / XGB)
print("="*72)
print("TRADITIONAL ML: Model_I (zero-shot) + Model_A (student GroupKFold OOF)")
print("="*72)

oof_mi = {m: {'prob': np.zeros(n_all), 'pred': np.zeros(n_all, dtype=int)}
          for m in ['LogisticRegression','DecisionTree','RandomForest','XGBoost']}
oof_ma = {m: {'prob': np.zeros(n_all), 'pred': np.zeros(n_all, dtype=int)}
          for m in ['LogisticRegression','DecisionTree','RandomForest','XGBoost']}

xgb_p = {
    'objective': 'binary:logistic', 'eval_metric': 'aucpr',
    'learning_rate': 0.05, 'max_depth': 4,
    'subsample': 0.8, 'colsample_bytree': 0.8,
    'seed': RANDOM_STATE, 'verbosity': 0,
}

print(f"\n{'Fold':>4}  {'TrPos':>5}  {'TePos':>5}  {'RF_F1':>7}  {'XGB_F1':>7}")
print("-"*40)

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_all, y_all, groups_all), 1):
    X_tr, X_va = X_all[tr_idx], X_all[va_idx]
    y_tr        = y_all[tr_idx]
    tr_pos      = int(y_tr.sum())
    va_pos      = int(y_all[va_idx].sum())
    neg_tr      = int((y_tr == 0).sum())
    spw         = round(neg_tr / max(tr_pos, 1), 2)

    # ── Model_I: zero-shot on student validation ───────────────────
    for mn in oof_mi:
        pipe = joblib.load(f"{P1_MODELS}/model_I_long_class_{mn}.joblib")
        oof_mi[mn]['prob'][va_idx]  = pipe.predict_proba(X_va)[:, 1]
        oof_mi[mn]['pred'][va_idx]  = (oof_mi[mn]['prob'][va_idx] >= 0.5).astype(int)

    # ── Model_A: train on student fold ────────────────────────────
    if tr_pos < 2:
        for mn in oof_ma:
            oof_ma[mn]['prob'][va_idx] = oof_mi['XGBoost']['prob'][va_idx]
            oof_ma[mn]['pred'][va_idx] = oof_mi['XGBoost']['pred'][va_idx]
        print(f"{fold:>4}  {tr_pos:>5}  {va_pos:>5}  {'fallbk':>7}  {'fallbk':>7}")
        continue

    sc   = StandardScaler().fit(X_tr)
    Xts  = sc.transform(X_tr)
    Xvs  = sc.transform(X_va)

    for mn, clf in [
        ('LogisticRegression', LogisticRegression(
            class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)),
        ('DecisionTree', DecisionTreeClassifier(
            max_depth=5, class_weight='balanced', random_state=RANDOM_STATE)),
        ('RandomForest', RandomForestClassifier(
            n_estimators=300, min_samples_leaf=1, class_weight='balanced',
            random_state=RANDOM_STATE, n_jobs=-1)),
    ]:
        clf.fit(Xts, y_tr)
        oof_ma[mn]['prob'][va_idx] = clf.predict_proba(Xvs)[:, 1]
        oof_ma[mn]['pred'][va_idx] = (oof_ma[mn]['prob'][va_idx] >= 0.5).astype(int)

    dtr = xgb.DMatrix(Xts, label=y_tr, feature_names=FEATURE_COLS)
    dva = xgb.DMatrix(Xvs,             feature_names=FEATURE_COLS)
    b   = xgb.train({**xgb_p, 'scale_pos_weight': spw}, dtr,
                    num_boost_round=100, verbose_eval=False)
    oof_ma['XGBoost']['prob'][va_idx] = b.predict(dva)
    oof_ma['XGBoost']['pred'][va_idx] = (oof_ma['XGBoost']['prob'][va_idx] >= 0.5).astype(int)

    rf_f1  = f1_score(y_all[va_idx], oof_ma['RandomForest']['pred'][va_idx], zero_division=0)
    xgb_f1 = f1_score(y_all[va_idx], oof_ma['XGBoost']['pred'][va_idx],      zero_division=0)
    print(f"{fold:>4}  {tr_pos:>5}  {va_pos:>5}  {rf_f1:>7.3f}  {xgb_f1:>7.3f}")

# ── Populate ALL_RESULTS ──────────────────────────────────────────────────────
for mn in oof_ma:
    ALL_RESULTS[f'Model_I_{mn}'] = {
        'prob': oof_mi[mn]['prob'], 'pred_05': oof_mi[mn]['pred'], 'category': 'Model_I'}
    ALL_RESULTS[f'Model_A_{mn}'] = {
        'prob': oof_ma[mn]['prob'], 'pred_05': oof_ma[mn]['pred'], 'category': 'Model_A'}

print(f"\nTraditional ML → ALL_RESULTS ({len(ALL_RESULTS)} entries)")
print(f"\n{'Model':30s}  {'PR-AUC':>7}  {'F1@0.5':>7}")
print("-"*48)
for mn in ['LogisticRegression','DecisionTree','RandomForest','XGBoost']:
    pr_a = average_precision_score(y_all, oof_ma[mn]['prob'])
    f1_a = f1_score(y_all, oof_ma[mn]['pred'], zero_division=0)
    print(f"  Model_A {mn:20s}  {pr_a:7.3f}  {f1_a:7.3f}")

TRADITIONAL ML: Model_I (zero-shot) + Model_A (student GroupKFold OOF)

Fold  TrPos  TePos    RF_F1   XGB_F1
----------------------------------------
   1    244     97    0.717    0.507
   2    284     57    0.495    0.466
   3    296     45    0.472    0.369
   4    264     77    0.383    0.511
   5    276     65    0.388    0.464

Traditional ML → ALL_RESULTS (12 entries)

Model                            PR-AUC   F1@0.5
------------------------------------------------
  Model_A LogisticRegression      0.370    0.443
  Model_A DecisionTree            0.317    0.433
  Model_A RandomForest            0.584    0.520
  Model_A XGBoost                 0.418    0.472


In [ ]:
# ── Cell 7b: RF + SMOTE variants ──────────────────────────────────────────────
# Finds best RF config (plain vs SMOTE) to feed into ensemble (Cell 11)
print("="*72)
print("RF + SMOTE variants  |  GroupKFold OOF")
print("="*72)

RF_SMOTE_STRATS = [0.1, 0.3, 0.5, 'auto']

for strat in RF_SMOTE_STRATS:
    oof_prob = np.zeros(n_all)
    oof_pred = np.zeros(n_all, dtype=int)

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_all, y_all, groups_all), 1):
        X_tr, X_va = X_all[tr_idx], X_all[va_idx]
        y_tr        = y_all[tr_idx]
        tr_pos      = int(y_tr.sum())
        tr_neg      = int((y_tr == 0).sum())
        if tr_pos < 2:
            continue

        sc  = StandardScaler().fit(X_tr)
        Xts = sc.transform(X_tr)
        Xvs = sc.transform(X_va)

        # safe ratio clamp


        k = min(3, tr_pos - 1) if tr_pos > 2 else 1
        if isinstance(strat, float):
            desired      = max(tr_pos + 1, int(tr_neg * strat))
            smote_strat  = {1: desired}
        else:
            smote_strat  = strat   # 'auto' passed through unchanged

        X_res, y_res = SMOTE(
            sampling_strategy=smote_strat,
            k_neighbors=k,
            random_state=RANDOM_STATE
        ).fit_resample(Xts, y_tr)

        # NO class_weight — SMOTE is doing the balancing
        rf = RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=1,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
        rf.fit(X_res, y_res)
        oof_prob[va_idx] = rf.predict_proba(Xvs)[:, 1]
        oof_pred[va_idx] = (oof_prob[va_idx] >= 0.5).astype(int)

    pr   = average_precision_score(y_all, oof_prob)
    f1   = f1_score(y_all, oof_pred, zero_division=0)
    prec = precision_score(y_all, oof_pred, zero_division=0)
    rec  = recall_score(y_all, oof_pred, zero_division=0)

    label = f'RF_SMOTE_{strat}'
    ALL_RESULTS[label] = {
        'prob':     oof_prob,
        'pred_05':  oof_pred,
        'category': 'Model_A+SMOTE'
    }
    print(f"  {label:<20}  PR-AUC={pr:.3f}  F1={f1:.3f}  "
          f"Prec={prec:.3f}  Rec={rec:.3f}")

# ── Show which RF config wins ─────────────────────────────────────────────────
rf_candidates = {k: v for k, v in ALL_RESULTS.items()
                 if k.startswith('RF_SMOTE') or k == 'Model_A_RandomForest'}

best_rf_label = max(rf_candidates,
                    key=lambda k: average_precision_score(
                        y_all, rf_candidates[k]['prob']))
best_rf_pr    = average_precision_score(y_all, rf_candidates[best_rf_label]['prob'])

print(f"\n{'─'*55}")
print(f"  Plain RF PR-AUC  : "
      f"{average_precision_score(y_all, ALL_RESULTS['Model_A_RandomForest']['prob']):.3f}")
for s in RF_SMOTE_STRATS:
    lbl = f'RF_SMOTE_{s}'
    pr  = average_precision_score(y_all, ALL_RESULTS[lbl]['prob'])
    mk  = ' ← BEST RF CONFIG' if lbl == best_rf_label else ''
    print(f"  RF+SMOTE({s}) PR-AUC: {pr:.3f}{mk}")

if best_rf_label == 'Model_A_RandomForest':
    print(f"\n  ← BEST RF CONFIG: Plain RF{mk}")

print(f"\nBEST_RF_LABEL = '{best_rf_label}'  (PR-AUC={best_rf_pr:.3f})")
print("This will be auto-picked by Cell 11 for ensemble.")

RF + SMOTE variants  |  GroupKFold OOF
  RF_SMOTE_0.1          PR-AUC=0.580  F1=0.513  Prec=0.603  Rec=0.446
  RF_SMOTE_0.3          PR-AUC=0.583  F1=0.517  Prec=0.515  Rec=0.519
  RF_SMOTE_0.5          PR-AUC=0.584  F1=0.524  Prec=0.487  Rec=0.566
  RF_SMOTE_auto         PR-AUC=0.580  F1=0.538  Prec=0.478  Rec=0.616

───────────────────────────────────────────────────────
  Plain RF PR-AUC  : 0.584
  RF+SMOTE(0.1) PR-AUC: 0.580
  RF+SMOTE(0.3) PR-AUC: 0.583
  RF+SMOTE(0.5) PR-AUC: 0.584
  RF+SMOTE(auto) PR-AUC: 0.580

  ← BEST RF CONFIG: Plain RF

BEST_RF_LABEL = 'Model_A_RandomForest'  (PR-AUC=0.584)
This will be auto-picked by Cell 11 for ensemble.


In [ ]:
# ── Cell 8: TrAdaBoost — Fine-tune Model_I XGB booster on student data ─────────
# Continues training from booster_I (industrial XGBoost) on each student fold.
# Tests boost rounds: [20, 50, 100, 200, 500]
print("="*72)
print("TrAdaBoost: Transfer from Model_I booster  |  rounds [20,50,100,200,500]")
print("="*72)

TRADA_ROUNDS = [20, 50, 100, 200, 500]

xgb_t = {
    'objective': 'binary:logistic', 'eval_metric': 'logloss',
    'learning_rate': 0.05, 'max_depth': 4,
    'subsample': 0.8, 'colsample_bytree': 0.8,
    'seed': RANDOM_STATE, 'verbosity': 0,
}

oof_tra = {r: {'prob': np.zeros(n_all), 'pred': np.zeros(n_all, dtype=int)}
           for r in TRADA_ROUNDS}

hdr = f"{'Fold':>4}  {'TrPos':>5}  {'TePos':>5}  " + "  ".join(f"r{r:>3d}" for r in TRADA_ROUNDS)
print(f"\n{hdr}")
print("-"*(len(hdr)+2))

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_all, y_all, groups_all), 1):
    X_tr, X_va = X_all[tr_idx], X_all[va_idx]
    y_tr        = y_all[tr_idx]
    tr_pos      = int(y_tr.sum())
    va_pos      = int(y_all[va_idx].sum())
    neg_tr      = int((y_tr == 0).sum())
    spw         = round(neg_tr / max(tr_pos, 1), 2)

    if tr_pos < 2:
        for r in TRADA_ROUNDS:
            oof_tra[r]['prob'][va_idx] = oof_mi['XGBoost']['prob'][va_idx]
            oof_tra[r]['pred'][va_idx] = oof_mi['XGBoost']['pred'][va_idx]
        print(f"{fold:>4}  {tr_pos:>5}  {va_pos:>5}  " + "  ".join(["  skip"]*len(TRADA_ROUNDS)))
        continue

    Xti = ind_scaler.transform(X_tr)
    Xvi = ind_scaler.transform(X_va)
    dtr = xgb.DMatrix(Xti, label=y_tr, feature_names=FEATURE_COLS)
    dval= xgb.DMatrix(Xvi,             feature_names=FEATURE_COLS)
    xgb_t['scale_pos_weight'] = spw

    f1s = []
    for r in TRADA_ROUNDS:
        b = xgb.train(xgb_t, dtr, num_boost_round=r,
                      xgb_model=booster_I, verbose_eval=False)
        oof_tra[r]['prob'][va_idx] = b.predict(dval)
        oof_tra[r]['pred'][va_idx] = (oof_tra[r]['prob'][va_idx] >= 0.5).astype(int)
        f1s.append(f"{f1_score(y_all[va_idx], oof_tra[r]['pred'][va_idx], zero_division=0):>6.3f}")
    print(f"{fold:>4}  {tr_pos:>5}  {va_pos:>5}  " + "  ".join(f1s))

# ── Best TrAda by PR-AUC ──────────────────────────────────────────────────────
best_r = max(TRADA_ROUNDS, key=lambda r: average_precision_score(y_all, oof_tra[r]['prob']))
print(f"\n{'rounds':>8}  {'PR-AUC':>8}  {'F1@0.5':>8}")
for r in TRADA_ROUNDS:
    pr = average_precision_score(y_all, oof_tra[r]['prob'])
    f1 = f1_score(y_all, oof_tra[r]['pred'], zero_division=0)
    mk = " ← best PR-AUC" if r == best_r else ""
    print(f"  +{r:>4d}r  {pr:>8.3f}  {f1:>8.3f}{mk}")

# ── Save best TrAda model (full data) ─────────────────────────────────────────
X_all_ind  = ind_scaler.transform(X_all)
spw_f      = round((y_all==0).sum() / max(y_all.sum(),1), 2)
xgb_t['scale_pos_weight'] = spw_f
dtr_all    = xgb.DMatrix(X_all_ind, label=y_all, feature_names=FEATURE_COLS)
b_tra_fin  = xgb.train(xgb_t, dtr_all, num_boost_round=best_r,
                        xgb_model=booster_I, verbose_eval=False)
b_tra_fin.save_model(f"{PT_MODELS}/model_TrAda_long_class_+{best_r}rounds.json")

# ── Populate ALL_RESULTS ──────────────────────────────────────────────────────
for r in TRADA_ROUNDS:
    ALL_RESULTS[f'TrAda_+{r}r'] = {
        'prob': oof_tra[r]['prob'], 'pred_05': oof_tra[r]['pred'], 'category': 'TrAda'}

print(f"\nTrAdaBoost → ALL_RESULTS ({len(ALL_RESULTS)} entries)")
print(f"Best model (rounds={best_r}) saved → {PT_MODELS}")

TrAdaBoost: Transfer from Model_I booster  |  rounds [20,50,100,200,500]

Fold  TrPos  TePos  r 20  r 50  r100  r200  r500
--------------------------------------------------
   1    244     97   0.479   0.519   0.528   0.560   0.566
   2    284     57   0.356   0.468   0.482   0.500   0.514
   3    296     45   0.294   0.368   0.387   0.407   0.418
   4    264     77   0.468   0.506   0.514   0.487   0.490
   5    276     65   0.413   0.453   0.485   0.505   0.541

  rounds    PR-AUC    F1@0.5
  +  20r     0.310     0.419
  +  50r     0.355     0.473
  + 100r     0.400     0.488
  + 200r     0.454     0.501
  + 500r     0.479     0.515 ← best PR-AUC

TrAdaBoost → ALL_RESULTS (21 entries)
Best model (rounds=500) saved → /content/drive/MyDrive/CodeSmell_Stage3/long_method/trAda/models


In [ ]:
# ── Cell 9: TabPFN — Baseline (No SMOTE) ────────────────────────────────────
print("="*72)
print("TabPFN: Baseline — No SMOTE  |  GroupKFold OOF")
print("="*72)

from tabpfn import TabPFNClassifier

oof_tab_base = np.zeros(n_all)

print(f"\n{'Fold':>4}  {'TrPos':>5}  {'TePos':>5}  {'PR-AUC':>8}  {'F1@0.5':>8}")
print("-"*40)

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_all, y_all, groups_all), 1):
    X_tr, X_va = X_all[tr_idx], X_all[va_idx]
    y_tr, y_va  = y_all[tr_idx], y_all[va_idx]
    tr_pos      = int(y_tr.sum())
    if tr_pos < 2:
        print(f"{fold:>4}  {tr_pos:>5}  {int(y_va.sum()):>5}  {'skip':>8}")
        continue
    sc  = StandardScaler().fit(X_tr)
    mdl = TabPFNClassifier()
    mdl.fit(sc.transform(X_tr), y_tr)
    prob = mdl.predict_proba(sc.transform(X_va))[:, 1]
    oof_tab_base[va_idx] = prob
    pr = average_precision_score(y_va, prob)
    f1 = f1_score(y_va, (prob >= 0.5).astype(int), zero_division=0)
    print(f"{fold:>4}  {tr_pos:>5}  {int(y_va.sum()):>5}  {pr:>8.3f}  {f1:>8.3f}")

pr_all = average_precision_score(y_all, oof_tab_base)
f1_all = f1_score(y_all, (oof_tab_base>=0.5).astype(int), zero_division=0)
print(f"\nOVERALL  PR-AUC={pr_all:.3f}  F1@0.5={f1_all:.3f}")

ALL_RESULTS['TabPFN_NoSMOTE'] = {
    'prob':     oof_tab_base,
    'pred_05':  (oof_tab_base >= 0.5).astype(int),
    'category': 'TabPFN'
}
print(f"TabPFN baseline → ALL_RESULTS ({len(ALL_RESULTS)} entries)")

TabPFN: Baseline — No SMOTE  |  GroupKFold OOF

Fold  TrPos  TePos    PR-AUC    F1@0.5
----------------------------------------


tabpfn-v2.6-classifier-v2.6_default.ckpt:   0%|          | 0.00/43.0M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

   1    244     97     0.448     0.125
   2    284     57     0.572     0.420
   3    296     45     0.535     0.451
   4    264     77     0.584     0.429
   5    276     65     0.536     0.362

OVERALL  PR-AUC=0.534  F1@0.5=0.345
TabPFN baseline → ALL_RESULTS (22 entries)


In [ ]:
# ── Cell 10: TabPFN + SMOTE variants (0.1, 0.3, 0.5, 0.8, auto) + SMOTE-ENN ──
print("="*72)
print("TabPFN + SMOTE  |  strategies: 0.1 / 0.3 / 0.5 / 0.8 / auto")
print("TabPFN + SMOTE-ENN  |  strategies: 0.3 / 0.5 / auto")
print("="*72)

from tabpfn import TabPFNClassifier
def run_tabpfn_with_resampler(resampler_name, strategy, use_enn=False):
    oof = np.zeros(n_all)
    for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_all, y_all, groups_all), 1):
        X_tr, X_va = X_all[tr_idx], X_all[va_idx]
        y_tr, y_va  = y_all[tr_idx], y_all[va_idx]
        tr_pos      = int(y_tr.sum())
        tr_neg      = int((y_tr == 0).sum())
        if tr_pos < 2:
            continue
        sc   = StandardScaler().fit(X_tr)
        Xts  = sc.transform(X_tr)
        Xvs  = sc.transform(X_va)
        k    = min(3, tr_pos - 1) if tr_pos > 2 else 1

        # ── Dict format bypasses float validator entirely ──────────────────────
        if isinstance(strategy, float):
            desired = max(tr_pos + 1, int(tr_neg * strategy))  # always > current count
            smote_strategy = {1: desired}
        else:
            smote_strategy = strategy   # 'auto' passed through unchanged
        # ───────────────────────────────────────────────────────────────────────

        smote_obj = SMOTE(sampling_strategy=smote_strategy, k_neighbors=k, random_state=RANDOM_STATE)
        if use_enn:
            resampler = SMOTEENN(smote=smote_obj, random_state=RANDOM_STATE)
        else:
            resampler = smote_obj
        X_res, y_res = resampler.fit_resample(Xts, y_tr)
        mdl = TabPFNClassifier()
        mdl.fit(X_res, y_res)
        oof[va_idx] = mdl.predict_proba(Xvs)[:, 1]
    return oof

print(f"\n{'Configuration':<40}  {'PR-AUC':>7}  {'F1@0.5':>7}  {'Prec@0.5':>9}  {'Rec@0.5':>8}")
print("-"*75)

SMOTE_STRATEGIES = [0.1, 0.3, 0.5, 0.8, 'auto']

# SMOTE only
for strat in SMOTE_STRATEGIES:
    label = f"TabPFN_SMOTE_{strat}"
    prob  = run_tabpfn_with_resampler(f"SMOTE({strat})", strat, use_enn=False)
    pred  = (prob >= 0.5).astype(int)
    pr    = average_precision_score(y_all, prob)
    f1    = f1_score(y_all, pred, zero_division=0)
    prec  = precision_score(y_all, pred, zero_division=0)
    rec   = recall_score(y_all, pred, zero_division=0)
    ALL_RESULTS[label] = {'prob': prob, 'pred_05': pred, 'category': 'TabPFN+SMOTE'}
    mk = " ← best" if strat == 0.5 else ""
    print(f"  {label:<38}  {pr:>7.3f}  {f1:>7.3f}  {prec:>9.3f}  {rec:>8.3f}{mk}")

print()

# SMOTE-ENN (noise-cleaned)
for strat in [0.3, 0.5, 'auto']:
    label = f"TabPFN_SMOTEENN_{strat}"
    prob  = run_tabpfn_with_resampler(f"SMOTEENN({strat})", strat, use_enn=True)
    pred  = (prob >= 0.5).astype(int)
    pr    = average_precision_score(y_all, prob)
    f1    = f1_score(y_all, pred, zero_division=0)
    prec  = precision_score(y_all, pred, zero_division=0)
    rec   = recall_score(y_all, pred, zero_division=0)
    ALL_RESULTS[label] = {'prob': prob, 'pred_05': pred, 'category': 'TabPFN+SMOTE-ENN'}
    print(f"  {label:<38}  {pr:>7.3f}  {f1:>7.3f}  {prec:>9.3f}  {rec:>8.3f}")

print(f"\nTabPFN+SMOTE variants → ALL_RESULTS ({len(ALL_RESULTS)} entries)")

TabPFN + SMOTE  |  strategies: 0.1 / 0.3 / 0.5 / 0.8 / auto
TabPFN + SMOTE-ENN  |  strategies: 0.3 / 0.5 / auto

Configuration                              PR-AUC   F1@0.5   Prec@0.5   Rec@0.5
---------------------------------------------------------------------------
  TabPFN_SMOTE_0.1                          0.527    0.328      0.564     0.232
  TabPFN_SMOTE_0.3                          0.606    0.518      0.522     0.513
  TabPFN_SMOTE_0.5                          0.597    0.506      0.504     0.507 ← best
  TabPFN_SMOTE_0.8                          0.577    0.481      0.484     0.478
  TabPFN_SMOTE_auto                         0.581    0.483      0.482     0.484

  TabPFN_SMOTEENN_0.3                       0.547    0.506      0.438     0.598
  TabPFN_SMOTEENN_0.5                       0.559    0.491      0.395     0.648
  TabPFN_SMOTEENN_auto                      0.562    0.489      0.377     0.698

TabPFN+SMOTE variants → ALL_RESULTS (30 entries)


In [ ]:
# ── Cell 11: Ensemble — dynamic best components per smell ─────────────────────
print("="*72)
print("ENSEMBLE: Best RF + Best XGB + Best TabPFN  |  Stacking + Voting")
print("="*72)

from tabpfn import TabPFNClassifier

# ── AUTO-SELECT best RF and best TabPFN from ALL_RESULTS ──────────────────────
# RF: plain RF vs all RF+SMOTE variants
rf_candidates  = {k: v for k, v in ALL_RESULTS.items()
                  if k.startswith('RF_SMOTE') or k == 'Model_A_RandomForest'}
best_rf_key    = max(rf_candidates,
                     key=lambda k: average_precision_score(
                         y_all, rf_candidates[k]['prob']))

# TabPFN: any TabPFN+SMOTE variant
tab_candidates = {k: v for k, v in ALL_RESULTS.items()
                  if 'TabPFN' in k}
best_tab_key   = max(tab_candidates,
                     key=lambda k: average_precision_score(
                         y_all, tab_candidates[k]['prob']))

print(f"  RF  component  : {best_rf_key}  "
      f"(PR-AUC={average_precision_score(y_all, rf_candidates[best_rf_key]['prob']):.3f})")
print(f"  TabPFN component: {best_tab_key}  "
      f"(PR-AUC={average_precision_score(y_all, tab_candidates[best_tab_key]['prob']):.3f})")
print(f"  XGB component  : Model_A_XGBoost  "
      f"(PR-AUC={average_precision_score(y_all, ALL_RESULTS['Model_A_XGBoost']['prob']):.3f})")
print()

# ── Re-run OOF for the 3 chosen components ────────────────────────────────────
# (use stored OOF probs directly — no need to retrain)
oof_rf_e  = ALL_RESULTS[best_rf_key]['prob']
oof_xgb_e = ALL_RESULTS['Model_A_XGBoost']['prob']
oof_tab_e = ALL_RESULTS[best_tab_key]['prob']

# ── Stacking (LR meta-model) ──────────────────────────────────────────────────
X_meta      = np.column_stack([oof_rf_e, oof_xgb_e, oof_tab_e])
meta_model  = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
meta_model.fit(X_meta, y_all)
prob_stack  = meta_model.predict_proba(X_meta)[:, 1]
pred_stack  = (prob_stack >= 0.5).astype(int)

# ── Soft voting (simple average) ──────────────────────────────────────────────
prob_vote = (oof_rf_e + oof_xgb_e + oof_tab_e) / 3.0
pred_vote = (prob_vote >= 0.5).astype(int)

ens_label  = f'Ensemble_Stacking_{best_rf_key}_+_{best_tab_key}'
vote_label = f'Ensemble_Voting_{best_rf_key}_+_{best_tab_key}'

ALL_RESULTS[ens_label]  = {'prob': prob_stack, 'pred_05': pred_stack,
                            'category': 'Ensemble'}
ALL_RESULTS[vote_label] = {'prob': prob_vote,  'pred_05': pred_vote,
                            'category': 'Ensemble'}

for name, prob, pred in [(ens_label,  prob_stack, pred_stack),
                          (vote_label, prob_vote,  pred_vote)]:
    pr = average_precision_score(y_all, prob)
    f1 = f1_score(y_all, pred, zero_division=0)
    print(f"  {name}")
    print(f"    PR-AUC={pr:.3f}  F1={f1:.3f}")

print(f"\nEnsembles → ALL_RESULTS ({len(ALL_RESULTS)} entries)")

ENSEMBLE: Best RF + Best XGB + Best TabPFN  |  Stacking + Voting
  RF  component  : Model_A_RandomForest  (PR-AUC=0.584)
  TabPFN component: TabPFN_SMOTE_0.3  (PR-AUC=0.606)
  XGB component  : Model_A_XGBoost  (PR-AUC=0.418)

  Ensemble_Stacking_Model_A_RandomForest_+_TabPFN_SMOTE_0.3
    PR-AUC=0.605  F1=0.468
  Ensemble_Voting_Model_A_RandomForest_+_TabPFN_SMOTE_0.3
    PR-AUC=0.611  F1=0.549

Ensembles → ALL_RESULTS (32 entries)


In [ ]:
# ── Cell 12: Master Results Table ─────────────────────────────────────────────
# Columns:
#   Model | Category | PR-AUC | Opt_Thresh | F1@opt | Prec@opt | Rec@opt |
#   TP | FP | FN | F1@0.5 | Prec@0.5 | Rec@0.5 | Δ_vs_BestRule
# Sort: PR-AUC desc (primary), F1@opt desc (secondary)
# ★ BEST marks row #1
print("="*120)
print(" LONG METHOD DETECTION — MASTER RESULTS TABLE")
print(f" Positives: {n_pos_all} / {n_all} ({n_pos_all/n_all*100:.1f}%)  |  GroupKFold(n={N_SPLITS})")
print("="*120)

def opt_threshold(y_true, y_prob):
    prec_c, rec_c, thr_c = precision_recall_curve(y_true, y_prob)
    f1_c   = 2 * prec_c * rec_c / (prec_c + rec_c + 1e-9)
    best_i = int(np.argmax(f1_c[:-1]))
    return float(thr_c[best_i])

def metrics_dual(y_true, prob, pred_05):
    pr_auc   = average_precision_score(y_true, prob)
    opt_thr  = opt_threshold(y_true, prob)
    pred_opt = (prob >= opt_thr).astype(int)

    # At optimal threshold
    f1_o   = f1_score(y_true, pred_opt, zero_division=0)
    pr_o   = precision_score(y_true, pred_opt, zero_division=0)
    re_o   = recall_score(y_true, pred_opt, zero_division=0)
    cm_o   = confusion_matrix(y_true, pred_opt)
    tp_o   = cm_o[1,1]; fp_o = cm_o[0,1]; fn_o = cm_o[1,0]

    # At 0.5
    f1_5   = f1_score(y_true, pred_05, zero_division=0)
    pr_5   = precision_score(y_true, pred_05, zero_division=0)
    re_5   = recall_score(y_true, pred_05, zero_division=0)

    return dict(pr_auc=round(pr_auc,3), opt_thr=round(opt_thr,3),
                f1_o=round(f1_o,3), pr_o=round(pr_o,3), re_o=round(re_o,3),
                tp=int(tp_o), fp=int(fp_o), fn=int(fn_o),
                f1_5=round(f1_5,3), pr_5=round(pr_5,3), re_5=round(re_5,3))

rows = []
for name, d in ALL_RESULTS.items():
    m = metrics_dual(y_all, d['prob'], d['pred_05'])
    rows.append({'Model': name, 'Category': d['category'], **m})

df_all = pd.DataFrame(rows)

# vs_BestRule baseline (PR-AUC delta)
best_rule_pr = df_all[df_all['Category']=='Rule']['pr_auc'].max()
df_all['delta_rule'] = (df_all['pr_auc'] - best_rule_pr).round(3)

# Sort: PR-AUC desc, F1@opt desc
df_all = df_all.sort_values(['pr_auc','f1_o'], ascending=[False,False]).reset_index(drop=True)

# Save
df_all.to_csv(f"{PA_RESULTS}/long_class_all_results.csv", index=False)

# ── Print ─────────────────────────────────────────────────────────────────────
CATEGORY_ORDER = ['Rule','Model_I','Model_A','TrAda','TabPFN',
                   'TabPFN+SMOTE','TabPFN+SMOTE-ENN','Ensemble']

W = 121
print(f"\n{'#':>2}  {'Model':<34} {'Category':>16}  "
      f"{'PR-AUC':>7}  {'OT':>6}  "
      f"{'F1@OT':>6} {'P@OT':>6} {'R@OT':>6}  "
      f"{'TP':>3} {'FP':>3} {'FN':>3}  "
      f"{'F1@.5':>6} {'P@.5':>5} {'R@.5':>5}  {'Δ_Rule':>7}")
print("─"*W)

prev_cat = None
for i, row in df_all.iterrows():
    cat = row['Category']
    if prev_cat is not None and cat != prev_cat:
        print("─"*W)
    prev_cat = cat
    star = " ★ BEST" if i == 0 else ""
    print(f"{i+1:>2}  {row['Model']:<34} {cat:>16}  "
          f"{row['pr_auc']:>7.3f}  {row['opt_thr']:>6.3f}  "
          f"{row['f1_o']:>6.3f} {row['pr_o']:>6.3f} {row['re_o']:>6.3f}  "
          f"{row['tp']:>3d} {row['fp']:>3d} {row['fn']:>3d}  "
          f"{row['f1_5']:>6.3f} {row['pr_5']:>5.3f} {row['re_5']:>5.3f}  "
          f"{row['delta_rule']:>+7.3f}{star}")

print("─"*W)
best = df_all.iloc[0]
print(f"\n🏆  WINNER : {best['Model']}")
print(f"    PR-AUC  : {best['pr_auc']:.3f}  (Δ vs best rule = {best['delta_rule']:+.3f})")
print(f"    Opt Thr : {best['opt_thr']:.3f}  →  F1={best['f1_o']:.3f}  "
      f"Prec={best['pr_o']:.3f}  Rec={best['re_o']:.3f}  TP={best['tp']}  FP={best['fp']}  FN={best['fn']}")
print(f"    @0.5    : F1={best['f1_5']:.3f}  Prec={best['pr_5']:.3f}  Rec={best['re_5']:.3f}")
print(f"\nTable saved → {PA_RESULTS}/long_class_all_results.csv")
# ── Save full results to CSV ───────────────────────────────────────────────────
csv_path = f"{PA_RESULTS}/{TARGET.replace('label_','')}_all_results.csv"
df_all.to_csv(csv_path, index=False)
print(f"\nFull results CSV saved → {csv_path}")
print(f"  {len(df_all)} models  |  columns: {list(df_all.columns)}")

 LONG METHOD DETECTION — MASTER RESULTS TABLE
 Positives: 341 / 3266 (10.4%)  |  GroupKFold(n=5)

 #  Model                                      Category   PR-AUC      OT   F1@OT   P@OT   R@OT   TP  FP  FN   F1@.5  P@.5  R@.5   Δ_Rule
─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
 1  Ensemble_Voting_Model_A_RandomForest_+_TabPFN_SMOTE_0.3         Ensemble    0.611   0.513   0.558  0.506  0.622  212 207 129   0.549 0.489 0.628   +0.371 ★ BEST
─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
 2  TabPFN_SMOTE_0.3                       TabPFN+SMOTE    0.606   0.416   0.542  0.487  0.610  208 219 133   0.518 0.522 0.513   +0.366
─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
 3  Ensemble_Stacking_Model_A_RandomForest_+_TabPFN_SMOTE_0.3         Ensemble    0.605   0.307  

In [ ]:
# ── Cell 13: Save all deployment files ─────────────────────────────────────────
print("="*72)
print("SAVING DEPLOYMENT FILES")
print("="*72)

from tabpfn import TabPFNClassifier

best_name   = df_all.iloc[0]['Model']
best_cfg    = df_all.iloc[0].to_dict()
best_cat    = best_cfg['Category']

# ── Final scaler (trained on full student data) ────────────────────────────────
sc_final    = StandardScaler().fit(X_all)
X_all_sc    = sc_final.transform(X_all)
joblib.dump(sc_final, f"{DEPLOY_DIR}/final_scaler.joblib")
print(f"  Scaler saved              → {DEPLOY_DIR}/final_scaler.joblib")

# ── Retrain best model on FULL student data ───────────────────────────────────
if 'TabPFN' in best_cat:
    strat = 0.5  # best SMOTE strategy
    k     = min(3, int(y_all.sum()) - 1)
    X_res, y_res = SMOTE(sampling_strategy=strat, k_neighbors=k,
                          random_state=RANDOM_STATE).fit_resample(X_all_sc, y_all)
    final_model = TabPFNClassifier()
    final_model.fit(X_res, y_res)
    model_path = f"{DEPLOY_DIR}/final_model_tabpfn_smote05.pkl"
    with open(model_path, 'wb') as fh:
        pickle.dump(final_model, fh)
    print(f"  TabPFN+SMOTE(0.5) model   → {model_path}")
else:
    # Best traditional model (RF)
    rf_fin = RandomForestClassifier(n_estimators=300, min_samples_leaf=1,
                                     class_weight='balanced',
                                     random_state=RANDOM_STATE, n_jobs=-1)
    rf_fin.fit(X_all_sc, y_all)
    model_path = f"{DEPLOY_DIR}/final_model_rf.joblib"
    joblib.dump(rf_fin, model_path)
    print(f"  RandomForest model saved  → {model_path}")

# ── Save all Model_A variants (full data) ─────────────────────────────────────
for mn, clf in [
    ('LogisticRegression', LogisticRegression(
        class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)),
    ('DecisionTree',       DecisionTreeClassifier(
        max_depth=5, class_weight='balanced', random_state=RANDOM_STATE)),
    ('RandomForest',       RandomForestClassifier(
        n_estimators=300, min_samples_leaf=1, class_weight='balanced',
        random_state=RANDOM_STATE, n_jobs=-1)),
]:
    clf.fit(X_all_sc, y_all)
    p = f"{PA_MODELS}/model_A_long_class_{mn}.joblib"
    joblib.dump(clf, p)
joblib.dump(sc_final, f"{PA_MODELS}/student_scaler.joblib")
print(f"  Model_A variants saved    → {PA_MODELS}")

# ── XGBoost Model_A final ─────────────────────────────────────────────────────
spw_f   = round((y_all==0).sum() / max(y_all.sum(),1), 2)
xgb_fin = {'objective':'binary:logistic','eval_metric':'aucpr',
            'learning_rate':0.05,'max_depth':4,'subsample':0.8,
            'colsample_bytree':0.8,'scale_pos_weight':spw_f,
            'seed':RANDOM_STATE,'verbosity':0}
dtr_f   = xgb.DMatrix(X_all_sc, label=y_all, feature_names=FEATURE_COLS)
b_fin   = xgb.train(xgb_fin, dtr_f, num_boost_round=100, verbose_eval=False)
b_fin.save_model(f"{PA_MODELS}/model_A_long_class_XGBoost.json")
print(f"  Model_A XGBoost saved     → {PA_MODELS}/model_A_long_class_XGBoost.json")

# ── Save complete results table ───────────────────────────────────────────────
results_path = f"{DEPLOY_DIR}/long_class_all_results.csv"
df_all.to_csv(results_path, index=False)
print(f"  Full results table        → {results_path}")

# ── Save deployment config JSON ───────────────────────────────────────────────
config = {
    'smell':                   'long_method',
    'best_model_name':         best_name,
    'best_model_category':     best_cat,
    'feature_cols':            FEATURE_COLS,
    'target':                  TARGET,
    'smote_strategy':          0.5,
    'n_student_samples':       n_all,
    'n_positives':             n_pos_all,
    'positive_rate':           round(n_pos_all / n_all, 4),
    'cv_method':               f'GroupKFold(n={N_SPLITS})',
    'best_rule_pr_auc':        round(float(df_all[df_all['Category']=='Rule']['pr_auc'].max()), 3),
    'pr_auc':                  best_cfg['pr_auc'],
    'optimal_threshold':       best_cfg['opt_thr'],
    'f1_at_optimal_threshold': best_cfg['f1_o'],
    'precision_at_optimal':    best_cfg['pr_o'],
    'recall_at_optimal':       best_cfg['re_o'],
    'tp_at_optimal':           int(best_cfg['tp']),
    'fp_at_optimal':           int(best_cfg['fp']),
    'fn_at_optimal':           int(best_cfg['fn']),
    'f1_at_0_5':               best_cfg['f1_5'],
    'precision_at_0_5':        best_cfg['pr_5'],
    'recall_at_0_5':           best_cfg['re_5'],
    'delta_vs_best_rule':      float(best_cfg['delta_rule']),
}
cfg_path = f"{DEPLOY_DIR}/long_class_final_config.json"
Path(cfg_path).write_text(json.dumps(config, indent=2))
print(f"  Deployment config JSON    → {cfg_path}")

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n" + "="*72)
print("DEPLOYMENT COMPLETE")
print("="*72)
print(f"  Directory   : {DEPLOY_DIR}")
print(f"  Best model  : {best_name}")
print(f"  PR-AUC      : {best_cfg['pr_auc']:.3f}")
print(f"  Threshold   : {best_cfg['opt_thr']:.3f}  (use this, not 0.5)")
print(f"  F1 @ thresh : {best_cfg['f1_o']:.3f}   (Prec={best_cfg['pr_o']:.3f}  Rec={best_cfg['re_o']:.3f})")
print(f"  F1 @ 0.5    : {best_cfg['f1_5']:.3f}   (Prec={best_cfg['pr_5']:.3f}  Rec={best_cfg['re_5']:.3f})")
print(f"  Δ vs rules  : {best_cfg['delta_rule']:+.3f}")
print("="*72)
print("Files saved:")
print(f"  final_scaler.joblib")
print(f"  final_model_tabpfn_smote05.pkl  (or final_model_rf.joblib)")
print(f"  model_A_long_class_*.joblib  (LR / DT / RF / XGB)")
print(f"  model_TrAda_long_class_+*rounds.json")
print(f"  long_class_all_results.csv")
print(f"  long_class_final_config.json")

SAVING DEPLOYMENT FILES
  Scaler saved              → /content/drive/MyDrive/CodeSmell_Stage3/long_method/deployment/final_scaler.joblib
  RandomForest model saved  → /content/drive/MyDrive/CodeSmell_Stage3/long_method/deployment/final_model_rf.joblib
  Model_A variants saved    → /content/drive/MyDrive/CodeSmell_Stage3/long_method/modelA/models
  Model_A XGBoost saved     → /content/drive/MyDrive/CodeSmell_Stage3/long_method/modelA/models/model_A_long_class_XGBoost.json
  Full results table        → /content/drive/MyDrive/CodeSmell_Stage3/long_method/deployment/long_class_all_results.csv
  Deployment config JSON    → /content/drive/MyDrive/CodeSmell_Stage3/long_method/deployment/long_class_final_config.json

DEPLOYMENT COMPLETE
  Directory   : /content/drive/MyDrive/CodeSmell_Stage3/long_method/deployment
  Best model  : Ensemble_Voting_Model_A_RandomForest_+_TabPFN_SMOTE_0.3
  PR-AUC      : 0.611
  Threshold   : 0.513  (use this, not 0.5)
  F1 @ thresh : 0.558   (Prec=0.506  Rec=0.62

In [ ]:
print('h')